# End-to-End Multimodal Sentiment Analysis Notebook

This notebook allows you to fine-tune a multimodal sentiment analysis model and run inference on your own video files. The process is divided into the following steps:
1.  **Setup and Installation:** Clone the repository and install the required libraries.
2.  **Data and Model Preparation:** Upload your pre-trained model and test videos.
3.  **Fine-Tuning (Optional):** Fine-tune the model on the CMU-MOSEI dataset.
4.  **Inference:** Run inference on your own `.mp4` video files.

## 1. Setup and Installation

### 1.1. Clone the Repository

In [ ]:
!git clone https://github.com/JugalGajjar/Multimodal-Sentiment-Analysis-with-MOSEI-Dataset.git
%cd Multimodal-Sentiment-Analysis-with-MOSEI-Dataset

### 1.2. Install Dependencies

In [ ]:
!pip install -r requirements.txt

### 1.3. Install Additional Libraries for Video/Audio Processing

In [ ]:
!pip install moviepy SpeechRecognition pydub librosa opencv-python-headless ffmpeg-python

### 1.4. Import Libraries

In [ ]:
import os
import sys
import subprocess
import numpy as np
import torch
from google.colab import files
import moviepy.editor as mp
import speech_recognition as sr
from pydub import AudioSegment
import librosa
import cv2
from tqdm import tqdm
from pathlib import Path

# Add project root to Python path
sys.path.append(os.getcwd())

# Import from the cloned repo
from config import *
from src.models.fusion import TransformerFusionModel
from transformers import BertTokenizer, BertModel

## 2. Data and Model Preparation

### 2.1. Upload Pre-trained Model

Upload your fine-tuned model checkpoint file (`.pt` or `.pth`). The best model from the original repository is `multimodal_fusion_best.pt`.

In [ ]:
# Create a directory for the model if it doesn't exist
model_dir = Path('models')
model_dir.mkdir(exist_ok=True)

# Upload the model file
print("Please upload your model checkpoint file (.pt or .pth)")
uploaded = files.upload()

# Move the uploaded file to the models directory
for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
  # Clear the directory before moving
  for p in model_dir.glob("*"):
      p.unlink()
  Path(fn).rename(model_dir / fn)

# Verify the file is in the correct location
model_path = next(model_dir.glob("*.pt"), None)
if model_path:
    print(f"Model successfully uploaded to: {model_path}")
else:
    print("Model file not found. Please make sure you uploaded a .pt file.")

### 2.2. Upload Test Videos

Create a directory for your test videos and upload your `.mp4` files.

In [ ]:
# Create a directory for test videos
test_videos_dir = Path('/content/test_videos')
test_videos_dir.mkdir(exist_ok=True)

# Go to the directory to upload files there
%cd /content/test_videos

# Upload video files
print(f"Please upload your video files (.mp4) into the '{test_videos_dir.name}' directory.")
uploaded_videos = files.upload()

# Verify upload
for fn in uploaded_videos.keys():
    print(f'User uploaded file "{fn}" with length {len(uploaded_videos[fn])} bytes')

# Go back to the project directory
%cd /content/Multimodal-Sentiment-Analysis-with-MOSEI-Dataset

# List the uploaded videos
video_files = list(test_videos_dir.glob("*.mp4"))
if video_files:
    print("\nUploaded video files:")
    for video_file in video_files:
        print(video_file)
else:
    print("\nNo video files found. Please make sure you uploaded .mp4 files to /content/test_videos/")

## 3. Fine-Tuning (Optional)

This section allows you to fine-tune the multimodal model on the CMU-MOSEI dataset. You can skip this section if you only want to run inference with the pre-trained model.

### 3.1. Download and Preprocess the CMU-MOSEI Dataset

In [ ]:
from src.data.download import install_mmsdk, download_mosei
from src.data.preprocess import MOSEIPreprocessor

# Install the CMU-MultimodalSDK
install_mmsdk()

# Download the MOSEI dataset
download_mosei(RAW_DATA_DIR, DATASET_NAME, DATASET_URL)

# Preprocess the dataset
preprocessor = MOSEIPreprocessor(RAW_DATA_DIR, PROCESSED_DATA_DIR)
preprocessor.process_dataset()
preprocessor.save_processed_data()

### 3.2. Set Up Fine-Tuning Parameters

In [ ]:
# Training parameters (you can adjust these)
BATCH_SIZE = 32
LEARNING_RATE = 1e-5
NUM_EPOCHS = 5 # Keep it small for a quick fine-tuning example
WEIGHT_DECAY = 1e-6
CHECKPOINT_PATH = 'models/multimodal_fusion_best.pt' # Path to the model you uploaded

### 3.3. Run Fine-Tuning

In [ ]:
from src.data.dataset import get_dataloaders
from src.training.trainer import Trainer
import torch.optim as optim

# Get dataloaders
dataloaders = get_dataloaders(batch_size=BATCH_SIZE)

# Initialize model
model = TransformerFusionModel(
    text_dim=TEXT_EMBEDDING_DIM,
    audio_dim=AUDIO_FEATURE_SIZE,
    visual_dim=VISUAL_FEATURE_SIZE,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_TRANSFORMER_LAYERS,
    num_heads=NUM_ATTENTION_HEADS,
    dropout_rate=DROPOUT_RATE
)
model = model.to(DEVICE)

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])

# Setup optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Initialize trainer
trainer = Trainer(
    model=model,
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    test_loader=dataloaders['test'],
    optimizer=optimizer,
    device=DEVICE,
    log_dir=LOGS_DIR,
    model_dir=MODELS_DIR
)

# Train the model
trainer.train(num_epochs=NUM_EPOCHS)

## 4. Inference on Custom Videos

This section runs the fine-tuned model on your own video files.

### 4.1. Video Preprocessing Function

In [ ]:
def preprocess_video(video_path, tokenizer, bert_model):
    # 1. Extract Audio
    video = mp.VideoFileClip(str(video_path))
    audio_path = 'temp_audio.wav'
    video.audio.write_audiofile(audio_path)

    # 2. Transcribe Audio to Text
    r = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio_data = r.record(source)
    try:
        text = r.recognize_google(audio_data)
    except sr.UnknownValueError:
        text = ""

    # 3. Extract Text Features (BERT)
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        text_features = bert_model(**inputs).last_hidden_state.mean(dim=1)

    # 4. Extract Audio Features (MFCCs)
    y, sr_ = librosa.load(audio_path, sr=None)
    mfccs = librosa.feature.mfcc(y=y, sr=sr_, n_mfcc=AUDIO_FEATURE_SIZE)
    audio_features = np.mean(mfccs, axis=1)

    # 5. Extract Visual Features (Averaged Frames)
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    if frames:
        avg_frame = np.mean(frames, axis=0)
        # Resize to match expected visual feature size
        visual_features = cv2.resize(avg_frame, (VISUAL_FEATURE_SIZE, 1)).flatten()
    else:
        visual_features = np.zeros(VISUAL_FEATURE_SIZE)
    
    # Clean up temp audio file
    os.remove(audio_path)
    
    return text_features, torch.tensor(audio_features).float().unsqueeze(0), torch.tensor(visual_features).float().unsqueeze(0)

### 4.2. Inference Function

In [ ]:
def run_inference(video_path, model, tokenizer, bert_model):
    # Preprocess the video to get features
    text_features, audio_features, visual_features = preprocess_video(video_path, tokenizer, bert_model)

    # Move features to the correct device
    text_features = text_features.to(DEVICE)
    audio_features = audio_features.to(DEVICE)
    visual_features = visual_features.to(DEVICE)

    # Run the model
    with torch.no_grad():
        prediction = model(text_features, audio_features, visual_features)
    
    return prediction.item()

### 4.3. Run Inference on Test Videos

In [ ]:
# Load the fine-tuned model
model = TransformerFusionModel(
    text_dim=TEXT_EMBEDDING_DIM,
    audio_dim=AUDIO_FEATURE_SIZE,
    visual_dim=VISUAL_FEATURE_SIZE,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_TRANSFORMER_LAYERS,
    num_heads=NUM_ATTENTION_HEADS,
    dropout_rate=DROPOUT_RATE
)
model = model.to(DEVICE)
model_path = next(Path('models').glob('*.pt'))
checkpoint = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load BERT model and tokenizer for preprocessing
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(DEVICE)
bert_model.eval()

# Get the list of uploaded videos
test_videos_dir = Path('/content/test_videos')
video_files = list(test_videos_dir.glob("*.mp4"))

# Run inference on each video
for video_file in video_files:
    print(f"Running inference on: {video_file.name}")
    sentiment_score = run_inference(video_file, model, tokenizer, bert_model)
    print(f"  Sentiment Score: {sentiment_score:.4f} (-3 to +3)")


## 5. Conclusion

This notebook provided a complete end-to-end workflow for multimodal sentiment analysis. You have learned how to:
- Set up the environment and install dependencies.
- Fine-tune a pre-trained model on the CMU-MOSEI dataset.
- Preprocess your own video files to extract text, audio, and visual features.
- Run inference on your videos to get a sentiment score.

You can further improve the performance by using more sophisticated feature extraction methods, especially for the visual modality (e.g., using OpenFace or other facial expression recognition models).